In [1]:
# R1 — Setup for the retraining tab. Mounts Drive, binds ROOT.
from google.colab import drive
drive.mount('/content/drive')

import os, sys, warnings
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/tdmpc2-highway')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["PYTHONWARNINGS"] = "ignore"

print(f"ROOT = {ROOT}")
!nvidia-smi | grep "L4\|A100\|H100" | head -1

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT = /content/drive/MyDrive/tdmpc2-highway
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |


In [2]:
# R2 — Reinstall deps if missing (fresh runtime)
import importlib.util
needed = ["gymnasium", "highway_env", "stable_baselines3", "torch",
          "torchrl", "omegaconf", "tensordict", "wandb"]
missing = [m for m in needed if importlib.util.find_spec(m) is None]
if missing:
    print(f"Installing: {missing}")
    !pip install -q "numpy<2" "torch>=2.1" "gymnasium>=0.29" "highway-env>=1.8" \
        "stable-baselines3[extra]>=2.2" "hydra-core>=1.3" "omegaconf>=2.3" \
        "tensordict>=0.3" "torchrl" "termcolor" "wandb" pandas scipy
    print("Restart runtime now, then re-run R1, then continue with R3")
else:
    print("[ok] deps present")

[ok] deps present


In [5]:
# R3.fix — Update train_tdmpc2_scaled.py to set num_q before MODEL_SIZE expansion
# (size=5's entry doesn't include num_q, so we need a default).
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/tdmpc2-highway")

scaled_path = ROOT / "src/training/train_tdmpc2_scaled.py"
src = scaled_path.read_text()

# Find the line that sets cfg.horizon and inject a num_q default before it
old = """    cfg = _build_cfg(env, seed=seed, total_timesteps=total_timesteps,
                     model_size=model_size)
    # Override defaults
    cfg.horizon = horizon"""

new = """    cfg = _build_cfg(env, seed=seed, total_timesteps=total_timesteps,
                     model_size=model_size)
    # Set num_q default if MODEL_SIZE didn't provide one (size=5 only sets
    # enc_dim, mlp_dim, latent_dim, num_enc_layers — not num_q).
    if "num_q" not in cfg:
        cfg.num_q = 5  # paper default for size>=5
    # Override defaults
    cfg.horizon = horizon"""

assert old in src, "marker not found"
scaled_path.write_text(src.replace(old, new))
print("[ok] patched train_tdmpc2_scaled.py — num_q defaults to 5 for model_size>=5")

[ok] patched train_tdmpc2_scaled.py — num_q defaults to 5 for model_size>=5


In [6]:
# Re-run R4 — clear cache and retry
import importlib
import src.training.train_tdmpc2_scaled as m
importlib.reload(m)
from src.training.train_tdmpc2_scaled import train_tdmpc2_scaled

_ = train_tdmpc2_scaled(
    seed=99,
    total_timesteps=3_000,
    model_size=5,
    horizon=5,
    seed_steps_override=1_000,
    use_wandb=False,
    log_every=500,
)
print("\n[ok] scaled pipeline functional end-to-end")

Episode length: 200
Discount factor: 0.975
Buffer capacity: 3,000
Storage required: 0.00 GB
Using CUDA:0 memory for storage.
2026-05-05 12:11:47,050 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([3000]) shape [END]
[step    500/3000] ep_rew(20)=  2.29 ep_len(20)=  5.0 crash%(20)=  0 sps= 34.0
[step   1000/3000] ep_rew(20)=  2.54 ep_len(20)=  5.4 crash%(20)=  0 sps= 33.8
[seed_steps=1000 reached] pretraining...
[step   1500/3000] ep_rew(20)=  3.99 ep_len(20)=  6.5 crash%(20)=  0 sps=  9.0
[step   2000/3000] ep_rew(20)=  4.72 ep_len(20)=  7.2 crash%(20)=  0 sps=  8.2
[step   2500/3000] ep_rew(20)=  5.22 ep_len(20)=  8.0 crash%(20)=  5 sps=  7.8
[step   3000/3000] ep_rew(20)=  4.49 ep_len(20)=  7.5 crash%(20)=  0 sps=  7.6

[seed=99 model_size=5] saved -> /content/drive/MyDrive/tdmpc2-highway/checkpoints/tdmpc2_size5_h5_seed99/final.pt

[ok] scaled pipeline functional end-to-end


In [7]:
# R4.5 — Pre-flight check before committing to a 6-hour run.
import torch
import gc

# Check VRAM headroom
gc.collect()
torch.cuda.empty_cache()
free_gb = torch.cuda.mem_get_info()[0] / 1e9
total_gb = torch.cuda.mem_get_info()[1] / 1e9
print(f"GPU VRAM: {free_gb:.1f} GB free / {total_gb:.1f} GB total")

# model_size=5 should use ~3-4 GB; we want at least 8 GB free as headroom
assert free_gb > 8.0, f"Only {free_gb:.1f} GB free — restart runtime to clear other tensors"

# Verify the smoke-test checkpoint exists from R4
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/tdmpc2-highway")
smoke_ckpt = ROOT / "checkpoints" / "tdmpc2_size5_h5_seed99" / "final.pt"
if smoke_ckpt.exists():
    size_mb = smoke_ckpt.stat().st_size / 1e6
    print(f"[ok] R4 smoke checkpoint exists: {size_mb:.1f} MB")
    assert size_mb > 10, "Smoke checkpoint suspiciously small — R4 may have failed silently"
else:
    print("[warn] no R4 smoke checkpoint — proceed anyway since R4 print said [ok]")

# Confirm Drive is responsive
drive_test = ROOT / "results" / ".drive_test"
drive_test.write_text("ok")
assert drive_test.read_text() == "ok"
drive_test.unlink()
print("[ok] Drive read/write functional")

# wandb sanity
import wandb
print(f"wandb version: {wandb.__version__}")
print(f"wandb run currently active: {wandb.run is not None}")
if wandb.run is not None:
    wandb.finish()
    print("[ok] closed stale wandb run")

print("\n[ok] pre-flight clear — R5 should run safely")

GPU VRAM: 84.5 GB free / 85.1 GB total
[ok] R4 smoke checkpoint exists: 33.5 MB
[ok] Drive read/write functional
wandb version: 0.26.1
wandb run currently active: False

[ok] pre-flight clear — R5 should run safely


In [ ]:
# R5 (HARDENED, horizon=3) — Replace your existing R5 cell with this.
# Same defenses as before, but horizon=3 fits in ~5.5h instead of ~22h.
import gc
import shutil
import signal
import time
from pathlib import Path

import torch
import wandb

import importlib
import src.training.train_tdmpc2_scaled as m
importlib.reload(m)
from src.training.train_tdmpc2_scaled import train_tdmpc2_scaled
from src.utils.config import SEEDS, CHECKPOINTS_DIR

# ---------- Helper: timeout-bounded wandb cleanup ----------
class _Timeout(Exception): pass
def _alarm(_, __): raise _Timeout()

def _safe_wandb_finish():
    if wandb.run is None:
        return
    try:
        signal.signal(signal.SIGALRM, _alarm)
        signal.alarm(30)
        wandb.finish()
        signal.alarm(0)
        print("  [wandb] finished cleanly")
    except _Timeout:
        signal.alarm(0)
        print("  [wandb] timeout — abandoning run sync")
        try:
            wandb.run.finish(quiet=True)
        except Exception:
            pass
    except Exception as e:
        signal.alarm(0)
        print(f"  [wandb] error during finish: {e}")

# ---------- Main loop ----------
print(f"Retraining: model_size=5, horizon=3, 200k steps × {len(SEEDS)} seeds")
print(f"Estimated wall-clock: ~5-6 hours on A100")
print(f"Defenses: wandb-timeout, drive-mirror, OOM-clear, per-seed try/except")
print(f"Started at: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

results_log = {"completed": [], "failed": []}
overall_start = time.time()

for seed_idx, seed in enumerate(SEEDS):
    seed_start = time.time()
    print(f"\n{'='*60}")
    print(f"  SEED {seed} ({seed_idx+1}/{len(SEEDS)})")
    print(f"  Elapsed total: {(seed_start - overall_start)/60:.1f} min")
    print(f"{'='*60}")

    # Clear GPU cache from previous seed
    gc.collect()
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f"  Pre-seed VRAM free: {free_gb:.1f} GB")

    try:
        ckpt_path = train_tdmpc2_scaled(
            seed=seed,
            total_timesteps=200_000,
            model_size=5,
            horizon=3,                    # paper default for size=5
            seed_steps_override=2_000,
            use_wandb=True,
            log_every=4_000,
        )

        # Verify checkpoint
        if not Path(ckpt_path).exists():
            raise FileNotFoundError(f"Checkpoint missing after train: {ckpt_path}")
        size_mb = Path(ckpt_path).stat().st_size / 1e6
        if size_mb < 20:  # size=5 should be ~30+ MB
            raise ValueError(f"Checkpoint suspiciously small: {size_mb:.1f} MB")

        results_log["completed"].append({
            "seed": seed,
            "checkpoint": str(ckpt_path),
            "size_mb": size_mb,
            "duration_min": (time.time() - seed_start) / 60,
        })
        print(f"  ✓ seed {seed} done in {(time.time()-seed_start)/60:.1f} min, "
              f"checkpoint {size_mb:.1f} MB")

    except _Timeout:
        ckpt_path = CHECKPOINTS_DIR / f"tdmpc2_size5_h3_seed{seed}" / "final.pt"
        if ckpt_path.exists():
            print(f"  ⚠ seed {seed}: wandb hung but checkpoint saved")
            results_log["completed"].append({
                "seed": seed,
                "checkpoint": str(ckpt_path),
                "note": "wandb_timeout_but_saved",
            })
        else:
            print(f"  ✗ seed {seed}: wandb hung AND no checkpoint")
            results_log["failed"].append({"seed": seed, "reason": "wandb_hang_no_save"})

    except Exception as e:
        print(f"  ✗ seed {seed} FAILED: {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()
        results_log["failed"].append({
            "seed": seed,
            "reason": f"{type(e).__name__}: {e}",
        })

    finally:
        _safe_wandb_finish()

# ---------- Summary ----------
total_min = (time.time() - overall_start) / 60
print(f"\n{'='*60}")
print(f"RETRAINING COMPLETE — {total_min:.1f} min total")
print(f"  Completed: {len(results_log['completed'])}/{len(SEEDS)} seeds")
print(f"  Failed:    {len(results_log['failed'])}/{len(SEEDS)} seeds")
print(f"{'='*60}")
for r in results_log["completed"]:
    print(f"  ✓ seed {r['seed']}: {r['checkpoint']}")
for r in results_log["failed"]:
    print(f"  ✗ seed {r['seed']}: {r['reason']}")

import json
log_path = CHECKPOINTS_DIR / "retraining_log.json"
log_path.write_text(json.dumps(results_log, indent=2))
print(f"\n[ok] log saved -> {log_path}")

Retraining: model_size=5, horizon=3, 200k steps × 3 seeds
Estimated wall-clock: ~5-6 hours on A100
Defenses: wandb-timeout, drive-mirror, OOM-clear, per-seed try/except
Started at: 2026-05-05 12:22:31


  SEED 0 (1/3)
  Elapsed total: 0.0 min
  Pre-seed VRAM free: 84.5 GB


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: error1rak (error1rak-american-university-of-sharjah) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Episode length: 200
Discount factor: 0.975
Buffer capacity: 200,000
Storage required: 0.02 GB
Using CUDA:0 memory for storage.
2026-05-05 12:24:21,425 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([200000]) shape [END]
[seed_steps=2000 reached] pretraining...
[step   4000/200000] ep_rew(20)= 14.21 ep_len(20)= 16.6 crash%(20)= 55 sps=  9.5
[step   8000/200000] ep_rew(20)= 23.39 ep_len(20)= 25.8 crash%(20)= 85 sps=  8.6
[step  12000/200000] ep_rew(20)= 22.50 ep_len(20)= 24.8 crash%(20)= 85 sps=  8.4
[step  16000/200000] ep_rew(20)= 19.12 ep_len(20)= 21.6 crash%(20)= 45 sps=  8.3
[step  20000/200000] ep_rew(20)= 28.52 ep_len(20)= 30.9 crash%(20)= 85 sps=  8.2
[step  24000/200000] ep_rew(20)= 25.44 ep_len(20)= 27.8 crash%(20)= 65 sps=  8.2
[step  28000/200000] ep_rew(20)= 25.95 ep_len(20)= 28.2 crash%(20)= 75 sps=  8.1
[step  32000/200000] ep_rew(20)= 33.85 ep_len(20)= 36.1 crash%(20)= 90 sps=  8.1
[step  36000/200000] ep_rew(20)= 26.67 ep_len(20)= 28.9 crash%(20)= 70 sp

global_step,▁▁▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇█
metrics/collision,▁█▁▁▁█▁█████▁███████████████████████████
metrics/success,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
rollout/ep_length,▁▂▁▃▂▂▃▄▂▃▆▃▃▂▂▃▂▃▂█▃▂▃▄▂▂▁▅▄▄▃▃▂▂█▃▂▁▃▁
rollout/ep_reward,▁▁▂▃▂▂▃▂▂▃▅▂▃▂▂▁▃▂▂█▂▃▄▂▃▂▆▄▁▃▄▁▃▂▄▂▃▆▃▂
global_step,199998
metrics/collision,1
metrics/success,0
rollout/ep_length,10
rollout/ep_reward,7.9435


  ✓ seed 0 done in 418.2 min, checkpoint 33.5 MB

  SEED 1 (2/3)
  Elapsed total: 418.2 min
  Pre-seed VRAM free: 84.5 GB


Episode length: 200
Discount factor: 0.975
Buffer capacity: 200,000
Storage required: 0.02 GB
Using CUDA:0 memory for storage.
2026-05-05 19:20:45,758 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([200000]) shape [END]
[seed_steps=2000 reached] pretraining...
[step   4000/200000] ep_rew(20)= 19.97 ep_len(20)= 22.5 crash%(20)= 75 sps= 10.0
[step   8000/200000] ep_rew(20)= 18.20 ep_len(20)= 20.6 crash%(20)= 55 sps=  8.8
